### **라이브러리 로드**

In [ ]:
# 데이터 처리
import pandas as pd
import numpy as np
from IPython.display import display

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

# 통계 검정
import scipy
from scipy.stats import chi2_contingency  # 카이제곱 검정
from scipy.stats import spearmanr         # 스피어만 상관계수
from scipy.stats import PermutationMethod # Monte Carlo 시뮬레이션
from scipy.stats import fisher_exact # Fisher's exact

# 경고 메시지 무시
import warnings
warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

# 데이터프레임 출력 제한 해제
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

---
### **데이터 로드**

In [ ]:
# FnB 롱폼에서 성공으로 분류된 영상 분류 결과
df_success_video = pd.read_csv("../../../data/results/main_dataset/롱폼 영상(+콘텐츠 분류 정보)/fnb_longform_success_data_with_classified.csv", encoding='utf-8')

# FnB 롱폼에서 실패로 분류된 영상 분류 결과
df_fail_video = pd.read_csv("../../../data/results/main_dataset/롱폼 영상(+콘텐츠 분류 정보)/fnb_longform_fail_data_with_classified.csv", encoding='utf-8')

# FnB 롱폼에서 성공으로 분류된 영상의 댓글 감성 분석 결과
df_success_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_fnb_longform_success_comment_final.csv", encoding='utf-8')

# FnB 롱폼에서 실패으로 분류된 영상의 댓글 감성 분석 결과
df_fail_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_fnb_longform_fail_comment.csv", encoding='utf-8')

In [ ]:
# 데이터 로드 결과 확인
print("성공 영상 분류 결과:", df_success_video.columns.tolist())
print("실패 영상 분류 결과:", df_fail_video.columns.tolist())
print("성공 영상의 댓글 감성 분석 결과:", df_success_comment.columns.tolist())
print("실패 영상의 댓글 감성 분석 결과:", df_fail_comment.columns.tolist())

In [ ]:
# 데이터프레임 확인
display(df_success_video.head(1))
display(df_fail_video.head(1))
display(df_success_comment.head(1))
display(df_fail_comment.head(1))

---
### **데이터 전처리**

In [ ]:
# ========================================
# [전처리 1] 필요한 컬럼만 선택
# ========================================

# [역할] 분석에 필요한 컬럼만 추출하여 데이터 크기 축소
# [근거] 불필요한 컬럼을 제거하여 메모리 효율화 및 가독성 향상

VIDEO_COLS = [
    'video_id', '조회수', '좋아요수', '댓글수', '영상길이(초)',
    '구독자수', '참여율(ER)', '일평균 조회수', 'score1', 'score2', 'grade',
    'cls_domain', 'cls_content_type', 'cls_marketing_purpose',
    'cls_cta_type', 'cls_is_series', 'cls_is_collaboration'
]

COMMENT_COLS = ['comment_id', 'video_id', 'sentiment', 'is_korean', 'is_event']

df_success_video_slim = df_success_video[VIDEO_COLS]
df_fail_video_slim = df_fail_video[VIDEO_COLS]
df_success_comment_slim = df_success_comment[COMMENT_COLS]
df_fail_comment_slim = df_fail_comment[COMMENT_COLS]

In [ ]:
# ========================================
# [전처리 2] 성공/실패 데이터 통합
# ========================================

# [역할] 성공/실패 영상 분류 결과와 댓글 감성 분석 결과를 각각 하나의 데이터프레임으로 통합
# [근거] 이후 분석에서 성공/실패를 grade 컬럼으로 구분하므로 하나로 합치는 게 효율적

df_video = pd.concat([df_success_video_slim, df_fail_video_slim], ignore_index=True)
df_comment_1 = pd.concat([df_success_comment_slim, df_fail_comment_slim], ignore_index=True)

print(f"영상 데이터: {len(df_video)}개") # 299개
print(f"댓글 데이터: {len(df_comment_1)}개") # 96860개

In [ ]:
# ========================================
# [전처리 3] 이벤트 참여 댓글 제거
# ========================================

# [역할] is_event=True인 댓글 제거
# [근거] 이벤트 참여 목적의 댓글은 영상 콘텐츠에 대한 반응이 아니므로
#        감성 비율을 왜곡할 수 있음

before = len(df_comment_1)
df_comment_2 = df_comment_1[df_comment_1['is_event'] == False]
print(f"이벤트 댓글 제거: {before - len(df_comment_2)}개 제거 → {len(df_comment_2)}개 남음") # 14710개 제거 → 82150개 남음

In [ ]:
# ========================================
# [전처리 4] 외국어 댓글 제거
# ========================================

# [역할] is_korean=False인 댓글 제거
# [근거] 외국어 댓글은 뉘앙스 파악의 한계로 감성 분류 신뢰도가 낮을 수 있음.
#        분석 대상이 한국 브랜드의 유튜브 채널이므로 한국어 댓글만 분석

before = len(df_comment_2)
df_comment = df_comment_2[df_comment_2['is_korean'] == True]
print(f"외국어 댓글 제거: {before - len(df_comment)}개 제거 → {len(df_comment)}개 남음") # 6090개 제거 → 76060개 남음

In [ ]:
# ========================================
# [전처리 5] video_id 기준으로 merge
# ========================================

# [역할] 영상 분류 정보에 댓글 감성 분석 결과를 결합
# [근거] 영상 데이터를 기준으로 merge하여 영상 분류 정보를 중심으로 댓글 데이터를 연결
# [작업] how='inner'로 양쪽 데이터에 모두 존재하는 video_id만 결합
#        → 댓글이 없는 영상은 제외됨

df = pd.merge(df_video, df_comment, on='video_id', how='inner')

print(f"merge 후 데이터: {len(df)}개") # 76060개
print(f"NaN 현황:\n{df.isna().sum()}")

In [ ]:
# ========================================
# [전처리 6] 최종 데이터 확인
# ========================================

# [역할] 전처리 완료 후 데이터 상태 최종 점검
# [근거] 행/열 수, 샘플 데이터를 확인하여 전처리 결과 검증

print(f"최종 데이터: {len(df)}행 / {len(df.columns)}열") # 38509행 / 21열
print(f"\n[감성 분포]")
print(df['sentiment'].value_counts())
print(f"\n[성공/실패 분포]")
print(df['grade'].value_counts())
display(df.head(3))

In [ ]:
# ========================================
# [전처리 7] 최종 데이터 확인
# ========================================

# [역할] 전처리 완료 후 데이터 상태 최종 점검
# [작업] 행/열 수, 감성 분포, 샘플 데이터 확인

print(f"최종 데이터: {len(df)}행 / {len(df.columns)}열")
print(f"\n[감성 분포]")
sentiment_counts = df['sentiment'].value_counts()
sentiment_pct = df['sentiment'].value_counts(normalize=True) * 100
for sentiment in sentiment_counts.index:
    print(f"  {sentiment}: {sentiment_counts[sentiment]}개 ({sentiment_pct[sentiment]:.1f}%)")

print(f"\n[성공/실패 분포]")
grade_counts = df['grade'].value_counts()
grade_pct = df['grade'].value_counts(normalize=True) * 100
for grade in grade_counts.index:
    print(f"  {grade}: {grade_counts[grade]}개 ({grade_pct[grade]:.1f}%)")

In [ ]:
# 성공/실패 데이터 분리
df_success = df[df['grade'] == '성공'].reset_index(drop=True)
df_fail = df[df['grade'] == '실패'].reset_index(drop=True)

print(f"성공 데이터: {len(df_success)}개")
print(f"실패 데이터: {len(df_fail)}개")

---
### **통계 검정(단일변수)**
성공으로 분류된 영상과 실패로 분류된 영상은 각각 통계 검정을 수행할 예정

##### **1단계: Cochran's rule 확인**
아래 두 조건을 모두 충족해야 카이제곱 검정을 적용할 수 있다.
- 기대빈도 1 미만인 셀이 없어야 함
- 기대빈도 5 미만인 셀의 비율이 20% 이하여야 함

두 조건 중 하나라도 위반 시 → Monte Carlo 시뮬레이션으로 대체 예정

##### **2단계: 카이제곱 검정 vs Monte Carlo 시뮬레이션**

| 구분 | 카이제곱 검정 | Monte Carlo 시뮬레이션 |
|---|---|---|
| 적용 조건 | Cochran's rule 충족 | Cochran's rule 위반 |
| 데이터 규모 | 대규모 | 대규모 |
| p-value 방식 | 이론적 분포 기반 | 시뮬레이션 기반 추정 |
| 계산 비용 | 낮음 | 높음 |

<p-value를 구하는 방법의 차이>
카이제곱 검정 → 카이제곱 통계량을 이론적 분포(카이제곱 분포)에 대입해서 p-value 계산
Monte Carlo → 동일한 카이제곱 통계량을 시뮬레이션 결과와 비교해서 p-value 추정

##### **3단계: 크래머 V (효과 크기)**
카이제곱/Monte Carlo 검정 결과가 유의미한 경우에만 적용한다.
카이제곱 통계량을 기반으로 계산하므로 두 검정 방법 모두 동일하게 적용 가능하다.
공식: `√(χ² / (N · min(r−1, c−1)))`
효과 크기 기준은 `df*(= min(r−1, c−1))`에 따라 다르게 적용한다.

| df* | 매우 작음 | 작음 | 중간 | 큼 |
|---|---|---|---|---|
| 1 | ~0.10 | 0.10~0.30 | 0.30~0.50 | 0.50~ |
| 2 | ~0.07 | 0.07~0.21 | 0.21~0.35 | 0.35~ |
| 3 | ~0.06 | 0.06~0.17 | 0.17~0.29 | 0.29~ |
| 4 | ~0.05 | 0.05~0.15 | 0.15~0.25 | 0.25~ |
| 5 | ~0.04 | 0.04~0.13 | 0.13~0.22 | 0.22~ |

##### **4단계: 조정된 잔차 사후검정**
카이제곱/Monte Carlo 검정 결과가 유의미한 경우에만 수행한다.
어떤 셀이 기대보다 유의미하게 많거나 적은지 파악한다.
- 기준: |조정된 잔차| > 1.96 (95% 신뢰수준)
- 양수: 기대보다 많음 (↑)
- 음수: 기대보다 적음 (↓)

In [ ]:
# ========================================
# 통계 검정 함수 정의
# ========================================

# [역할] 범주형 변수와 sentiment 간의 관계를 통계적으로 검정
# [작업] Cochran's rule 확인 → 카이제곱 or Monte Carlo 시뮬레이션
#        → 크래머 V → 조정된 잔차 사후검정

def chi2_test(df, col, target='sentiment', alpha=0.05, n_simulations=9999):
    """
    Parameters
    ----------
    df            : 분석 데이터프레임
    col           : 검정할 범주형 변수 컬럼명
    target        : 감성 컬럼명 (기본값: 'sentiment')
    alpha         : 유의수준 (기본값: 0.05)
    n_simulations : Monte Carlo 시뮬레이션 횟수 (기본값: 9999)
    """

    print(f"\n{'='*60}")
    print(f"[{col}] × [{target}] 검정")
    print(f"{'='*60}")

    # ── 1단계: 교차표 및 기대빈도 계산 ──────────────────────
    # [작업] pd.crosstab으로 교차표 생성
    #        chi2_contingency로 카이제곱 통계량, p-value, 자유도, 기대빈도 계산
    contingency_table = pd.crosstab(df[col], df[target])
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # ── 2단계: Cochran's rule 확인 ───────────────────────────
    # [작업] 조건 1: 기대빈도 1 미만인 셀이 없어야 함
    #       조건 2: 기대빈도 5 미만인 셀의 비율이 20% 이하여야 함
    #       -> 두 조건 중 하나라도 위반 시 Monte Carlo 시뮬레이션으로 바꿔서 수행
    total_cells = expected.size
    zero_cells = (expected < 1).sum()
    low_cells = (expected < 5).sum()
    low_cells_pct = low_cells / total_cells * 100

    print(f"\n[Cochran's rule 확인]")
    print(f"  전체 셀 수: {total_cells}")
    print(f"  기대빈도 1 미만 셀 수: {zero_cells}")
    print(f"  기대빈도 5 미만 셀 수: {low_cells} ({low_cells_pct:.1f}%)")

    # True: Monte Carlo 시뮬레이션 사용 / False: 카이제곱 검정 사용
    use_monte_carlo = (zero_cells > 0) or (low_cells_pct > 20)

    if use_monte_carlo:
        print(f"  ⚠️ Cochran's rule 위반 → Monte Carlo 시뮬레이션으로 대체")
    else:
        print(f"  ✅ Cochran's rule 충족 → 카이제곱 검정 진행")

    # ── 3단계: 카이제곱 검정 or Monte Carlo 시뮬레이션 ────────
    if use_monte_carlo:
        # [작업] scipy의 PermutationMethod를 사용하여 행/열 합계를 고정한 채
        #        n_simulations번 랜덤 교차표를 생성하여 p-value를 추정
        #        correction=False: Yates 연속성 보정은 Monte Carlo와 함께 사용 불가
        #        random_state=42: 재현 가능한 결과를 위해 시드 고정
        method = PermutationMethod(n_resamples=n_simulations, random_state=42)
        result = chi2_contingency(contingency_table, correction=False, method=method)
        chi2 = result.statistic
        p_val = result.pvalue

        print(f"\n[Monte Carlo 시뮬레이션 결과] (n={n_simulations:,})")
        print(f"  χ²     : {chi2:.4f}")
        print(f"  p-value: {p_val:.4f}")
        print(f"  자유도  : {dof}") # 자유도가 클수록 더 많은 범주 조합을 검정한다는 의미 
                                   # -> 카이제곱 통계량이 같아도 자유도에 따라 p-value가 달라진다고 함
    else:
        # [작업] p를 p_val로 명시적으로 할당하여 이후 코드에서 일관되게 사용
        p_val = p
        print(f"\n[카이제곱 검정 결과]")
        print(f"  χ²     : {chi2:.4f}")
        print(f"  p-value: {p_val:.4f}")
        print(f"  자유도  : {dof}")

    if p_val < alpha:
        print(f"  ✅ 유의미한 관계 있음 (p < {alpha})")
    else:
        print(f"  ❌ 유의미한 관계 없음 (p >= {alpha})")
        return None

    # ── 4단계: 크래머 V (효과 크기) ──────────────────────────
    # [작업] df* = min(r-1, c-1)에 따라 효과 크기 기준이 달라짐
    #        공식: √(χ² / (N · df*))
    #        df*별 효과 크기 기준은 Cohen(1988)의 w값(0.10, 0.30, 0.50)을
    #        V = w / √(df*) 공식에 대입하여 계산
    n = contingency_table.values.sum() # 전체 댓글 수
    df_star  = min(contingency_table.shape) - 1
    cramer_v = np.sqrt(chi2 / (n * df_star))

    small = 0.10 / np.sqrt(df_star)
    medium = 0.30 / np.sqrt(df_star)
    large = 0.50 / np.sqrt(df_star)

    print(f"\n[크래머 V (효과 크기)]")
    print(f"  df*    : {df_star}")
    print(f"  크래머 V: {cramer_v:.4f}")
    if cramer_v < small:
        print(f"  → 효과 매우 작음 ({small:.2f} 미만)")
    elif cramer_v < medium:
        print(f"  → 효과 작음 ({small:.2f} 이상 {medium:.2f} 미만)")
    elif cramer_v < large:
        print(f"  → 효과 중간 ({medium:.2f} 이상 {large:.2f} 미만)")
    else:
        print(f"  → 효과 큼 ({large:.2f} 이상)")

    # ── 5단계: 조정된 잔차 사후검정 ──────────────────────────
    # [작업] 조정된 잔차 = (관측빈도 - 기대빈도) / 표준오차
    #        표준오차 = sqrt(기대빈도 × (1 - 행비율) × (1 - 열비율))
    #        |조정된 잔차| > 1.96이면 유의미한 셀 (95% 신뢰수준)
    observed = contingency_table.values
    row_sums = observed.sum(axis=1, keepdims=True)
    col_sums = observed.sum(axis=0, keepdims=True)
    total = observed.sum()

    std_err = np.sqrt(expected * (1 - row_sums / total) * (1 - col_sums / total))
    adjusted_resid = (observed - expected) / std_err

    df_resid = pd.DataFrame(
        adjusted_resid,
        index=contingency_table.index,
        columns=contingency_table.columns
    ).round(2)

    print(f"\n[조정된 잔차 사후검정]")
    print(f"  기준: |조정된 잔차| > 1.96 → 유의미한 셀 (95% 신뢰수준)")
    display(df_resid)

    print(f"\n[유의미한 셀 요약]")
    for row in df_resid.index:
        for col_name in df_resid.columns:
            resid = df_resid.loc[row, col_name]
            if abs(resid) > 1.96:
                direction = "많음 (↑)" if resid > 0 else "적음 (↓)"
                print(f"  {row} × {col_name}: {resid} → 기대보다 {direction}")

    return {
        'chi2' : chi2,
        'p-value' : p_val,
        'df_star' : df_star,
        'cramer_v' : cramer_v,
        'use_monte_carlo': use_monte_carlo,
        'residuals' : df_resid
    }

In [ ]:
# ========================================
# 카이제곱 검정 실행 - 성공으로 분류된 영상
# ========================================

# [역할] 성공으로 분류된 영상의 범주형 변수와 sentiment 간의 관계 검정
# [작업] 각 컬럼에 대해 chi2_test 함수 실행

CATEGORICAL_COLS = [
    'cls_content_type',         # 영상 유형
    'cls_marketing_purpose',    # 영상 업로드 목적
    'cls_cta_type',             # CTA 타입
    'cls_is_series',            # 시리즈물 여부
    'cls_is_collaboration'      # 콜라보 여부
]

print("=" * 60)
print("✅ 성공으로 분류된 영상의 통계 검정 결과")
print("=" * 60)

results_success = {}
for col in CATEGORICAL_COLS:
    results_success[col] = chi2_test(df_success, col)

In [ ]:
# ========================================
# 카이제곱 검정 실행 - 실패로 분류된 영상
# ========================================

# [역할] 실패로 분류된 영상의 범주형 변수와 sentiment 간의 관계 검정
# [작업] 각 컬럼에 대해 chi2_test 함수 실행

CATEGORICAL_COLS = [
    'cls_content_type',         # 영상 유형
    'cls_marketing_purpose',    # 영상 업로드 목적
    'cls_cta_type',             # CTA 타입
    'cls_is_series',            # 시리즈물 여부
    'cls_is_collaboration'      # 콜라보 여부
]

print("=" * 60)
print("✅ 실패로 분류된 영상의 통계 검정 결과")
print("=" * 60)

results_fail = {}
for col in CATEGORICAL_COLS:
    results_fail[col] = chi2_test(df_fail, col)

---
### **통계 검정 결과 시각화**

In [ ]:
def visualize_chi2(df, col, result, target='sentiment'):
    """
    Parameters
    ----------
    df     : 분석 데이터프레임
    col    : 검정한 범주형 변수 컬럼명
    result : chi2_test 함수의 반환값 (dict)
    target : 감성 컬럼명 (기본값: 'sentiment')
    """

    if result is None:
        print(f"[{col}] 유의미한 관계가 없어 시각화를 생략합니다.")
        return

    # ── 원본 보호: 함수 진입 시 무조건 copy ─────────────────
    # [근거] bool 타입 여부와 관계없이 함수 내부 수정이
    #        원본 DataFrame에 영향을 주지 않도록 방어적으로 복사
    df1 = df.copy()

    SENTIMENT_ORDER  = ["긍정", "중립", "부정"]
    SENTIMENT_COLORS = {"긍정": "#4CAF50", "중립": "#9E9E9E", "부정": "#F44336"}

    # ── bool 타입 컬럼 문자열 변환 ───────────────────────────
    # [작업] bool 타입 컬럼을 문자열로 변환하여
    #        y축 레이블이 숫자(0, 1)로 나타나는 문제 방지
    if df1[col].dtype == bool:
        df1[col] = df1[col].map({True: 'True', False: 'False'})

    # ── 감성 비율 계산 ────────────────────────────────────────
    # [작업] 범주별 감성 비율 계산 후 긍정 비율 기준 오름차순 정렬
    grouped = (
        df1.groupby([col, target])
        .size()
        .reset_index(name='count')
    )
    total_per_group = grouped.groupby(col)['count'].transform('sum')
    grouped['ratio'] = grouped['count'] / total_per_group * 100

    pivot = (
        grouped
        .pivot_table(index=col, columns=target, values='ratio', fill_value=0)
        .reindex(columns=SENTIMENT_ORDER, fill_value=0)
    )

    # [작업] 긍정 비율 기준 오름차순 정렬
    pivot = pivot.sort_values('긍정', ascending=True)

    # ── 그래프 생성 ───────────────────────────────────────────
    n_rows = len(pivot)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, n_rows * 0.5 + 2)))
    fig.suptitle(f"[{col}] × [{target}] 감성 분석", fontsize=15, fontweight='bold', y=1.01)

    # ── 차트 1: 누적 가로 막대차트 ───────────────────────────
    ax1 = axes[0]
    left = np.zeros(n_rows)

    for sentiment in SENTIMENT_ORDER:
        values = pivot[sentiment].values
        bars   = ax1.barh(
            pivot.index,
            values,
            left=left,
            color=SENTIMENT_COLORS[sentiment],
            label=sentiment,
            height=0.6,
        )
        for bar, val in zip(bars, values):
            if val == 0:
                continue
            x_center = bar.get_x() + bar.get_width() / 2
            y_center = bar.get_y() + bar.get_height() / 2

            if val >= 5:  # 내부 표시
                ax1.text(x_center, y_center, f"{val:.1f}%",
                         ha='center', va='center',
                         fontsize=8, color='white', fontweight='bold')
            else:          # 외부에 선 연결 (겹침 방지: 위/아래 교차 배치)
                offset_sign = 1 if SENTIMENT_ORDER.index(sentiment) % 2 == 0 else -1
                ax1.annotate(
                    f"{val:.1f}%",
                    xy=(x_center, y_center),
                    xytext=(x_center, y_center + offset_sign * 0.45),
                    arrowprops=dict(arrowstyle="-", color="gray", lw=1),
                    ha='center', va='center',
                    fontsize=8, color='black',
                )
        left += values

    ax1.set_xlim(0, 100)
    ax1.set_xlabel("비율 (%)", fontsize=11)

    # [작업] y축 레이블에 각 범주별 댓글 수 추가
    count_per_group = df1.groupby(col).size()
    new_labels = [f"{idx}\n(n={count_per_group[idx]:,})" for idx in pivot.index]
    ax1.set_yticklabels(new_labels, fontsize=9)

    ax1.set_title("감성 비율 (긍정 비율 기준 정렬)", fontsize=13)
    ax1.legend(
        loc='upper left',
        bbox_to_anchor=(0, 1.00),   # y축 위쪽으로 올리기
        ncol=3,                      # 범례를 가로로 나열
        fontsize=10,
    )
    ax1.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d%%"))
    sns.despine(ax=ax1)

    # ── 차트 2: 조정된 잔차 히트맵 ──────────────────────────
    # [작업] RdYlGn 컬러맵 적용
    #        양수(초록): 해당 감성이 기대보다 자주 나타남
    #        음수(빨강): 해당 감성이 기대보다 적게 나타남
    #        |조정된 잔차| > 1.96인 셀만 유의미한 셀로 해석
    ax2 = axes[1]

    # [작업] bool 인덱스를 문자열로 변환 후 reindex하여 누적 막대차트와 순서 일치
    df1_resid = result['residuals'].copy()
    df1_resid.index = df1_resid.index.astype(str)
    df1_resid = df1_resid.reindex(pivot.index)

    sns.heatmap(
        df1_resid,
        ax=ax2,
        cmap='RdYlGn',
        center=0,
        annot=True,
        fmt='.2f',
        linewidths=0.5,
        cbar_kws={'label': '조정된 잔차'}
    )

    ax2.set_title("조정된 잔차 히트맵\n( |잔차| > 1.96, 유의미한 셀)\n(양수: 해당 감성이 기대보다 자주 나타남 / 음수: 해당 감성이 기대보다 적게 나타남)", fontsize=11)
    ax2.set_xlabel("감성", fontsize=11)
    ax2.set_ylabel("")

    plt.tight_layout()
    plt.show()

In [ ]:
# ========================================
# 성공으로 분류된 영상 데이터의 검정 결과 시각화
# ========================================

# [역할] 카이제곱 검정 결과를 시각화
# [작업] 각 컬럼에 대해 visualize_chi2 함수 실행

for col in CATEGORICAL_COLS:
    visualize_chi2(df_success, col, results_success[col])
    plt.show()

In [ ]:
# ========================================
# 실패로 분류된 영상 데이터의 검정 결과 시각화
# ========================================

# [역할] 카이제곱 검정 결과를 시각화
# [작업] 각 컬럼에 대해 visualize_chi2 함수 실행

for col in CATEGORICAL_COLS:
    visualize_chi2(df_fail, col, results_fail[col])
    plt.show()

---
### **두 가지 정보의 조합에서 댓글의 감성 분석하기 (1)**
- `content_type × is_series` 분석
    - is_series=True  → content_type별 긍정 비율 계산
    - is_series=False → content_type별 긍정 비율 계산
    - 각 content_type마다 시리즈일때의 긍정률 vs 시리즈가 아닐 때의 긍정률 비교
    - 시리즈가 유리한 콘텐츠 유형 / 시리즈로 제작하지 않는 것(단독)이 유리한 콘텐츠 유형 도출이 목표

In [ ]:
# ========================================
# [분석] content_type × is_series 교차 분석
# ========================================

# [역할] 성공/실패 영상 각각에서, 시리즈 여부에 따라
#        콘텐츠 유형별 댓글 긍정 비율이 달라지는지 분석
# [근거] 성공/실패를 분리하여 분석함으로써
#        "어떤 콘텐츠 유형을 시리즈로 만들면 긍정 반응을 얻는지"를
#        성과 그룹별로 독립적으로 도출할 수 있음

def analyze_content_type_x_is_series(df_input, group_label):
    """
    특정 성과 그룹(성공 또는 실패) 내에서
    content_type × is_series 교차 분석을 수행하는 함수

    Parameters
    ----------
    df_input : 분석 대상 데이터프레임 (df_success 또는 df_fail)
    group_label: 출력 및 파일명에 사용할 레이블 (예: '성공', '실패')
    """

    print(f"\n{'='*60}")
    print(f"[{group_label} 영상] content_type × is_series 분석")
    print(f"{'='*60}")

    # ── 0. 분석용 컬럼 추출 ─────────────────────────────────────
    # [작업] 분석에 필요한 컬럼만 추출
    df_cross = df_input[['video_id', 'cls_content_type', 'cls_is_series', 'sentiment']].copy()

    print(f"\n[cls_is_series 분포]")
    print(df_cross['cls_is_series'].value_counts())
    print(f"\n[cls_content_type 분포]")
    print(df_cross['cls_content_type'].value_counts())

    # ── 1. 긍정 여부 컬럼 생성 ──────────────────────────────────
    # [작업] sentiment == '긍정'이면 1, 아니면 0
    df_cross['is_positive'] = (df_cross['sentiment'] == '긍정').astype(int)

    # ── 2. (cls_content_type × cls_is_series)별 긍정 비율 집계 ──
    agg = (
        df_cross.groupby(['cls_content_type', 'cls_is_series'])['is_positive']
        .agg(positive_count='sum', total='count')
        .reset_index()
    )
    agg['positive_ratio'] = agg['positive_count'] / agg['total']

    print(f"\n=== (콘텐츠 유형 × 시리즈 여부)별 긍정 비율 ===")
    display(agg.sort_values(['cls_content_type', 'cls_is_series']).reset_index(drop=True))

    # ── 3. 피벗: 시리즈 vs 단독 나란히 비교 ──────────────────────
    # [작업] cls_is_series를 컬럼으로 피벗
    # [주의] 한쪽 조건(시리즈/단독)이 없는 content_type은 NaN으로 남음
    pivot = agg.pivot_table(
        index='cls_content_type',
        columns='cls_is_series',  # 컬럼명이 cls_is_series의 고유값인 True/False로 생성됨
        values='positive_ratio'
    ).reset_index()
    
    pivot.columns.name = None

    pivot = pivot.rename(columns={
        True: '시리즈_긍정률',
        False: '단독_긍정률'
    })

    # [작업] 차이 계산 후 유리한 형식 판별
    # 차이(시리즈-단독) > 0 : 시리즈_긍정률 > 단독_긍정률 → 시리즈가 유리한 콘텐츠 유형
    # 차이(시리즈-단독) < 0 : 단독_긍정률 > 시리즈_긍정률 → 단독이 유리한 콘텐츠 유형
    # [참고] 최종 결론 테이블은 차이 기준 내림차순 정렬이므로
    #        표 위쪽(양수) = 시리즈 유리, 표 아래쪽(음수) = 단독 유리
    pivot['차이(시리즈-단독)'] = pivot['시리즈_긍정률'] - pivot['단독_긍정률']
    pivot['해석'] = pivot['차이(시리즈-단독)'].apply(
        lambda x: '🟢 시리즈물일 때, 유리함' if x > 0 else ('🔵 시리즈물이 아닐 때, 유리함' if x < 0 else '➖ 동일')
    )

    # ── 4. 통계 검정 (카이제곱 or Fisher's exact) ─────────────────
    # [작업] 각 content_type 내에서 is_series와 sentiment(긍정/비긍정) 간
    #        연관성이 통계적으로 유의한지 검정
    stat_results = []

    for ctype in df_cross['cls_content_type'].unique():
        sub = df_cross[df_cross['cls_content_type'] == ctype]
        ct = pd.crosstab(sub['is_positive'], sub['cls_is_series'])

        n_series = int(sub['cls_is_series'].sum())
        n_non_series = int((~sub['cls_is_series']).sum())

        # 한쪽 조건이 없으면 검정 불가
        if ct.shape != (2, 2):
            stat_results.append({
                'content_type':  ctype,
                'n_series': n_series, # 시리즈물인 영상의 개수
                'n_non_series':  n_non_series, # 시리즈물이 아닌 영상의 개수
                'p_value': float('nan'),
                'effect_size(φ)': float('nan'),
                'test':'skip(한쪽조건없음)',
                'significant':   False
            })
            continue
        
        # 시리즈/단독 각각 최소 30개 미만이면 검정 불가
        # 이유: 샘플이 너무 작으면 p값이 유의해도 신뢰하기 어려움
        if n_series < 30 or n_non_series < 30:
            stat_results.append({
                'content_type':  ctype,
                'n_series': n_series,
                'n_non_series':  n_non_series,
                'p_value': float('nan'),
                'effect_size(φ)': float('nan'),
                'test':'skip(샘플부족)',
                'significant':   False
            })
            continue

        # 카이제곱으로 기대빈도를 먼저 계산하여 검정 방법을 결정
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            chi2, p, dof, expected = chi2_contingency(ct)

        if (expected < 5).any():
            # 기대빈도 < 5인 셀이 있으면 Fisher's exact로 교체
            # 이유: 샘플이 작을 때 카이제곱 근사값이 부정확해지므로
            #       정확한 확률을 직접 계산하는 Fisher's exact를 사용
            _, p = fisher_exact(ct)
            test_name = "Fisher's exact"
        else:
            test_name = "Chi-square"

        # 효과크기: 파이 계수(φ) 계산
        # φ = sqrt(χ² / n) → 2×2 표에서 두 범주형 변수 간 연관성의 강도
        # 한 가지 콘텐츠 유형마다 통계 검정을 수행하고 있기 때문에 2×2 표가 맞음
        # 해석 기준(Cohen's w): 0.1 미만(매우 작은 효과), 0.1~0.3(작은 효과), 0.3~0.5(중간 효과), 0.5 이상(큰 효과)
        n_total = sub['cls_is_series'].count()
        phi = round(np.sqrt(chi2 / n_total), 4)

        stat_results.append({
            'content_type': ctype,
            'n_series': n_series, # 시리즈물인 영상의 개수
            'n_non_series': n_non_series, # 시리즈물이 아닌 영상의 개수
            'p_value': round(p, 4),
            'effect_size(φ)': phi,
            'test': test_name,
            'significant': p < 0.05 # 통계적 유의성 
        })

    stat_df = pd.DataFrame(stat_results).sort_values('p_value')
    print(f"\n=== 통계 검정 결과 ===")
    display(stat_df.reset_index(drop=True))

    # ── 5. 최종 결론 테이블 ──────────────────────────────────────
    # [작업] 긍정률 피벗 + 검정 결과 합치기
    # [근거] 통계적으로 유의하지 않은 차이는 결론으로 채택하지 않음
    df_final = pivot.merge(
        stat_df[['content_type', 'p_value', 'effect_size(φ)', 'significant', 'test']],
        left_on='cls_content_type',
        right_on='content_type',
        how='left'
    ).drop(columns='content_type')

    df_final['결론'] = df_final.apply(
        lambda row: row['해석'] if row['significant'] else '⚠️ 통계적으로 유의하지 않음',
        axis=1
    )

    print(f"\n=== [{group_label}] 최종 분석 결론 ===")
    display(
        df_final[[
            'cls_content_type', '시리즈_긍정률', '단독_긍정률',
            '차이(시리즈-단독)', 'p_value', 'effect_size(φ)', '결론'
        ]]
        .sort_values('차이(시리즈-단독)', ascending=False)
        .reset_index(drop=True)
    )

    # ── 6. 시각화 ────────────────────────────────────────────────
    # 샘플 부족 또는 한쪽 조건 없음으로 skip된 항목은 그래프에서도 제외
    valid_types = stat_df[stat_df['significant'] | (stat_df['p_value'].notna())]['content_type'].tolist()
    plot_df = (
        df_final[df_final['cls_content_type'].isin(valid_types)]
        .set_index('cls_content_type')[['시리즈_긍정률', '단독_긍정률']]
        .dropna()
    )

    fig, ax = plt.subplots(figsize=(12, 5))

    x = np.arange(len(plot_df))
    width = 0.35

    ax.bar(x - width/2, plot_df['시리즈_긍정률'], width, label='시리즈', color='#4C72B0', alpha=0.85)
    ax.bar(x + width/2, plot_df['단독_긍정률'],   width, label='단독',   color='#DD8452', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=25, ha='right', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('긍정 댓글 비율')
    ax.set_title(f'[{group_label} 영상] 콘텐츠 유형별 시리즈 vs 단독 긍정 댓글 비율', fontsize=12)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

    return df_final

In [ ]:
# 성공 영상에서 시리즈가 유리한 콘텐츠 유형, 시리즈가 아닐 때 유리한 콘텐츠 유형 확인용
df_final_success_series = analyze_content_type_x_is_series(df_success, '성공')

In [ ]:
# 실패 영상에서 시리즈가 유리한 콘텐츠 유형, 시리즈가 아닐 때 유리한 콘텐츠 유형 확인용
df_final_fail_series = analyze_content_type_x_is_series(df_fail,'실패')

---
### **두 가지 정보의 조합에서 댓글의 감성 분석하기 (2)**
- `content_type × is_collaboration` 분석
    - is_collaboration=True → content_type별 긍정 비율 계산
    - is_collaboration=False → content_type별 긍정 비율 계산
    - 어떤 콘텐츠 유형에서 콜라보한 영상이 긍정적인 반응을 이끌어내기에 유리한지, 
    또 어떤 콘텐츠 유형에서 콜라보하지 않은 영상이 유리한지 확인하는 것이 목표임 

In [ ]:
# ========================================
# [분석] content_type × is_collaboration 교차 분석
# ========================================

# [역할] 성공/실패 영상 각각에서, 콜라보 여부에 따라
#        콘텐츠 유형별 댓글 긍정 비율이 달라지는지 분석
# [근거] 성공/실패를 분리하여 분석함으로써
#        "어떤 콘텐츠 유형을 콜라보로 만들면 긍정 반응을 얻는지"를
#        성과 그룹별로 독립적으로 도출할 수 있음

def analyze_content_type_x_is_collaboration(df_input, group_label):
    """
    특정 성과 그룹(성공 또는 실패) 내에서
    content_type × is_collaboration 교차 분석을 수행하는 함수

    Parameters
    ----------
    df_input   : 분석 대상 데이터프레임 (df_success 또는 df_fail)
    group_label: 출력 및 파일명에 사용할 레이블 (예: '성공', '실패')
    """

    print(f"\n{'='*60}")
    print(f"[{group_label} 영상] content_type × is_collaboration 분석")
    print(f"{'='*60}")

    # ── 0. 분석용 컬럼 추출 ─────────────────────────────────────
    # [작업] 분석에 필요한 컬럼만 추출
    df_cross = df_input[['video_id', 'cls_content_type', 'cls_is_collaboration', 'sentiment']].copy()

    print(f"\n[cls_is_collaboration 분포]")
    print(df_cross['cls_is_collaboration'].value_counts())
    print(f"\n[cls_content_type 분포]")
    print(df_cross['cls_content_type'].value_counts())

    # ── 1. 긍정 여부 컬럼 생성 ──────────────────────────────────
    # [작업] sentiment == '긍정'이면 1, 아니면 0
    df_cross['is_positive'] = (df_cross['sentiment'] == '긍정').astype(int)

    # ── 2. (cls_content_type × cls_is_collaboration)별 긍정 비율 집계 ──
    agg = (
        df_cross.groupby(['cls_content_type', 'cls_is_collaboration'])['is_positive']
        .agg(positive_count='sum', total='count')
        .reset_index()
    )
    agg['positive_ratio'] = agg['positive_count'] / agg['total']

    print(f"\n=== (콘텐츠 유형 × 콜라보 여부)별 긍정 비율 ===")
    display(agg.sort_values(['cls_content_type', 'cls_is_collaboration']).reset_index(drop=True))

    # ── 3. 피벗: 콜라보 vs 단독 나란히 비교 ──────────────────────
    # [작업] cls_is_collaboration을 컬럼으로 피벗
    # [주의] 한쪽 조건(콜라보/단독)이 없는 content_type은 NaN으로 남음
    pivot = agg.pivot_table(
        index='cls_content_type',
        columns='cls_is_collaboration',  # 컬럼명이 cls_is_collaboration의 고유값인 True/False로 생성됨
        values='positive_ratio'
    ).reset_index()

    pivot.columns.name = None

    pivot = pivot.rename(columns={
        True:  '콜라보_긍정률',
        False: '단독_긍정률'
    })

    # [작업] 차이 계산 후 유리한 형식 판별
    # 차이(콜라보-단독) > 0 : 콜라보_긍정률 > 단독_긍정률 → 콜라보가 유리한 콘텐츠 유형
    # 차이(콜라보-단독) < 0 : 단독_긍정률 > 콜라보_긍정률 → 단독이 유리한 콘텐츠 유형
    # [참고] 최종 결론 테이블은 차이 기준 내림차순 정렬이므로
    #        표 위쪽(양수) = 콜라보 유리, 표 아래쪽(음수) = 단독 유리
    pivot['차이(콜라보-단독)'] = pivot['콜라보_긍정률'] - pivot['단독_긍정률']
    pivot['해석'] = pivot['차이(콜라보-단독)'].apply(
        lambda x: '🟢 콜라보일 때, 유리함' if x > 0 else ('🔵 콜라보하지 않을 때, 유리함' if x < 0 else '➖ 동일')
    )

    # ── 4. 통계 검정 (카이제곱 or Fisher's exact) ─────────────────
    # [작업] 각 content_type 내에서 is_collaboration과 sentiment(긍정/비긍정) 간
    #        연관성이 통계적으로 유의한지 검정
    stat_results = []

    for ctype in df_cross['cls_content_type'].unique():
        sub = df_cross[df_cross['cls_content_type'] == ctype]
        ct = pd.crosstab(sub['is_positive'], sub['cls_is_collaboration'])

        n_collaboration     = int(sub['cls_is_collaboration'].sum())
        n_non_collaboration = int((~sub['cls_is_collaboration']).sum())

        # 한쪽 조건이 없으면 검정 불가
        if ct.shape != (2, 2):
            stat_results.append({
                'content_type':       ctype,
                'n_collaboration':     n_collaboration,     # 콜라보 영상의 댓글 수
                'n_non_collaboration': n_non_collaboration, # 단독 영상의 댓글 수
                'p_value':            float('nan'),
                'effect_size(φ)':     float('nan'),
                'test':               'skip(한쪽조건없음)',
                'significant':        False
            })
            continue

        # 콜라보/단독 각각 최소 30개 미만이면 검정 불가
        # 이유: 샘플이 너무 작으면 p값이 유의해도 신뢰하기 어려움
        if n_collaboration < 30 or n_non_collaboration < 30:
            stat_results.append({
                'content_type':       ctype,
                'n_collaboration':     n_collaboration,
                'n_non_collaboration': n_non_collaboration,
                'p_value':            float('nan'),
                'effect_size(φ)':     float('nan'),
                'test':               'skip(샘플부족)',
                'significant':        False
            })
            continue

        # 카이제곱으로 기대빈도를 먼저 계산하여 검정 방법을 결정
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            chi2, p, dof, expected = chi2_contingency(ct)

        if (expected < 5).any():
            # 기대빈도 < 5인 셀이 있으면 Fisher's exact로 교체
            # 이유: 샘플이 작을 때 카이제곱 근사값이 부정확해지므로
            #       정확한 확률을 직접 계산하는 Fisher's exact를 사용
            _, p = fisher_exact(ct)
            test_name = "Fisher's exact"
        else:
            test_name = "Chi-square"

        # 효과크기: 파이 계수(φ) 계산
        # φ = sqrt(χ² / n) → 2×2 표에서 두 범주형 변수 간 연관성의 강도
        # 한 가지 콘텐츠 유형마다 통계 검정을 수행하고 있기 때문에 2×2 표가 맞음
        # 해석 기준(Cohen's w): 0.1 미만(매우 작은 효과), 0.1~0.3(작은 효과), 0.3~0.5(중간 효과), 0.5 이상(큰 효과)
        n_total = sub['cls_is_collaboration'].count()
        phi = round(np.sqrt(chi2 / n_total), 4)

        stat_results.append({
            'content_type':       ctype,
            'n_collaboration':     n_collaboration,     # 콜라보 영상의 댓글 수
            'n_non_collaboration': n_non_collaboration, # 단독 영상의 댓글 수
            'p_value':            round(p, 4),
            'effect_size(φ)':     phi,
            'test':               test_name,
            'significant':        p < 0.05             # 통계적 유의성
        })

    stat_df = pd.DataFrame(stat_results).sort_values('p_value')
    print(f"\n=== 통계 검정 결과 ===")
    display(stat_df.reset_index(drop=True))

    # ── 5. 최종 결론 테이블 ──────────────────────────────────────
    # [작업] 긍정률 피벗 + 검정 결과 합치기
    # [근거] 통계적으로 유의하지 않은 차이는 결론으로 채택하지 않음
    df_final = pivot.merge(
        stat_df[['content_type', 'p_value', 'effect_size(φ)', 'significant', 'test']],
        left_on='cls_content_type',
        right_on='content_type',
        how='left'
    ).drop(columns='content_type')

    df_final['결론'] = df_final.apply(
        lambda row: row['해석'] if row['significant'] else '⚠️ 통계적으로 유의하지 않음',
        axis=1
    )

    print(f"\n=== [{group_label}] 최종 분석 결론 ===")
    display(
        df_final[[
            'cls_content_type', '콜라보_긍정률', '단독_긍정률',
            '차이(콜라보-단독)', 'p_value', 'effect_size(φ)', '결론'
        ]]
        .sort_values('차이(콜라보-단독)', ascending=False)
        .reset_index(drop=True)
    )

    # ── 6. 시각화 ────────────────────────────────────────────────
    # 샘플 부족 또는 한쪽 조건 없음으로 skip된 항목은 그래프에서도 제외
    valid_types = stat_df[stat_df['significant'] | (stat_df['p_value'].notna())]['content_type'].tolist()
    plot_df = (
        df_final[df_final['cls_content_type'].isin(valid_types)]
        .set_index('cls_content_type')[['콜라보_긍정률', '단독_긍정률']]
        .dropna()
    )

    fig, ax = plt.subplots(figsize=(12, 5))

    x = np.arange(len(plot_df))
    width = 0.35

    ax.bar(x - width/2, plot_df['콜라보_긍정률'], width, label='콜라보', color='#4C72B0', alpha=0.85)
    ax.bar(x + width/2, plot_df['단독_긍정률'],   width, label='단독',   color='#DD8452', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=25, ha='right', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('긍정 댓글 비율')
    ax.set_title(f'[{group_label} 영상] 콘텐츠 유형별 콜라보 vs 단독 긍정 댓글 비율', fontsize=12)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

    return df_final

In [ ]:
# 함수 실행 (성공한 영상)
final_success_collabo = analyze_content_type_x_is_collaboration(df_success, '성공')

In [ ]:
# 함수 실행 (실패한 영상)
final_fail_collabo = analyze_content_type_x_is_collaboration(df_fail, '실패')

---
### **두 가지 정보의 조합에서 댓글의 감성 분석하기 (3)**
- `content_type × marketing_purpose` 분석
    - 영상 업로드 목적(marketing_purpose)에 따라 어떤 유형의 영상을 제작하는 것이
    시청자의 긍정적인 반응을 이끌어내기에 유리한지 확인하는 것이 목표임

In [ ]:
# ================================================
# [분석] content_type × marketing_purpose 교차 분석
# ================================================

# [역할] 성공/실패 영상 각각에서, 마케팅 목적과 콘텐츠 유형의 조합이
#        댓글 긍정 비율에 미치는 영향을 분석
# [근거] 두 범주형 변수 간 전체 연관성을 카이제곱으로 검정하고,
#        조합별 긍정률을 히트맵으로 시각화하여
#        긍정 반응을 이끌어내기에 유리한 조합을 도출

def analyze_content_type_x_marketing_purpose(df_input, group_label):
    """
    특정 성과 그룹(성공 또는 실패) 내에서
    content_type × marketing_purpose 교차 분석을 수행하는 함수

    Parameters
    ----------
    df_input   : 분석 대상 데이터프레임 (df_success 또는 df_fail)
    group_label: 출력 및 파일명에 사용할 레이블 (예: '성공', '실패')
    """

    print(f"\n{'='*60}")
    print(f"[{group_label} 영상] content_type × marketing_purpose 분석")
    print(f"{'='*60}")

    # ── 0. 분석용 컬럼 추출 ─────────────────────────────────────
    # [작업] 분석에 필요한 컬럼만 추출
    df_cross = df_input[['video_id', 'cls_content_type', 'cls_marketing_purpose', 'sentiment']].copy()

    print(f"\n[cls_marketing_purpose 분포]")
    print(df_cross['cls_marketing_purpose'].value_counts())
    print(f"\n[cls_content_type 분포]")
    print(df_cross['cls_content_type'].value_counts())

    # ── 1. 긍정 여부 컬럼 생성 ──────────────────────────────────
    # [작업] sentiment == '긍정'이면 1, 아니면 0
    df_cross['is_positive'] = (df_cross['sentiment'] == '긍정').astype(int)

    # ── 2. (cls_content_type × cls_marketing_purpose)별 긍정 비율 집계 ──
    agg = (
        df_cross.groupby(['cls_content_type', 'cls_marketing_purpose'])['is_positive']
        .agg(positive_count='sum', total='count')
        .reset_index()
    )
    agg['positive_ratio'] = agg['positive_count'] / agg['total']

    print(f"\n=== (콘텐츠 유형 × 마케팅 목적)별 긍정 비율 ===")
    display(agg.sort_values(['cls_content_type', 'cls_marketing_purpose']).reset_index(drop=True))

    # ── 3. 통계 검정: 전체 교차표 카이제곱 ───────────────────────
    # [작업] content_type × marketing_purpose 전체 교차표로 카이제곱 검정
    # [근거] marketing_purpose가 9개 범주라 개별 검정이 아닌
    #        전체 연관성을 한 번에 검정하는 방식을 사용
    # [주의] 댓글 수(total) 기준으로 교차표 생성
    #        → 각 셀: 해당 조합의 전체 댓글 수

    # 전체 교차표 생성 (행: content_type, 열: marketing_purpose, 값: 댓글 수)
    ct = pd.crosstab(df_cross['cls_content_type'], df_cross['cls_marketing_purpose'])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        chi2, p, dof, expected = chi2_contingency(ct)

    # 효과크기: Cramér's V 계산
    # [근거] 카이제곱은 표 크기에 따라 값이 달라지므로
    #        표 크기를 보정한 Cramér's V를 효과크기로 사용
    # 계산식: V = sqrt(χ² / (n × (min(행수, 열수) - 1)))
    # 해석 기준(Cohen): 0.1 미만(매우 작은 효과), 0.1~0.3(작은 효과),
    #                   0.3~0.5(중간 효과), 0.5 이상(큰 효과) --> df*=1일때 기준
    n_total = df_cross.shape[0]
    min_dim = min(ct.shape) - 1
    cramers_v = round(np.sqrt(chi2 / (n_total * min_dim)), 4)

    print(f"\n=== 통계 검정 결과 ===")
    print(f"카이제곱 통계량 : {chi2:.4f}")
    print(f"p-value         : {p:.4f}")
    print(f"자유도          : {dof}")
    print(f"Cramér's V      : {cramers_v}")
    print(f"통계적 유의성   : {'유의함 (p < 0.05)' if p < 0.05 else '유의하지 않음 (p >= 0.05)'}")

    # ── 4. 피벗: 히트맵용 긍정률 테이블 생성 ─────────────────────
    # [작업] 행: content_type, 열: marketing_purpose, 값: positive_ratio
    # [주의] 댓글 수가 30개 미만인 셀은 NaN으로 처리하여 히트맵에서 제외
    #        → 샘플이 너무 작은 조합은 신뢰하기 어려움

    # 셀별 댓글 수 피벗 (30개 미만 필터링용)
    total_pivot = agg.pivot_table(
        index='cls_content_type',
        columns='cls_marketing_purpose',
        values='total'
    )

    # 긍정률 피벗
    ratio_pivot = agg.pivot_table(
        index='cls_content_type',
        columns='cls_marketing_purpose',
        values='positive_ratio'
    )

    # 댓글 수 30개 미만인 셀은 NaN으로 마스킹
    # [근거] 샘플이 너무 작으면 긍정률이 극단적으로 나올 수 있음
    ratio_pivot_masked = ratio_pivot.where(total_pivot >= 30)

    print(f"\n=== 조합별 긍정률 테이블 (댓글 수 30개 미만 셀은 NaN) ===")
    display(ratio_pivot_masked.round(4))

    # ── 5. 히트맵 시각화 ─────────────────────────────────────────
    # [작업] 조합별 긍정률을 히트맵으로 시각화
    # [근거] 수치 비교표보다 색상으로 패턴을 직관적으로 파악할 수 있음
    #        NaN 셀은 회색으로 표시 (데이터 부족)

    fig, ax = plt.subplots(figsize=(14, 7))

    # NaN 셀 위치 마스크 생성
    nan_mask = ratio_pivot_masked.isna()
    
    # 배경을 먼저 회색으로 깔기 (NaN 셀이 회색으로 보이도록)
    ax.set_facecolor('#D3D3D3')

    sns.heatmap(
        ratio_pivot_masked,
        annot=True,              # 셀 안에 수치 표시
        fmt='.2%',               # 퍼센트 형식
        cmap='RdYlGn',           # 빨강(낮음) → 노랑 → 초록(높음)
        vmin=0.7,                # 색상 범위 하한 (데이터에 따라 조정)
        vmax=1.0,                # 색상 범위 상한
        linewidths=0.5,
        linecolor='gray',
        mask=nan_mask,           # NaN 셀은 렌더링 제외 → 배경색(회색)이 드러남
        ax=ax,
        cbar_kws={'label': '긍정 댓글 비율'}
    )

    ax.set_title(
        f'[{group_label} 영상] 콘텐츠 유형 × 마케팅 목적별 긍정 댓글 비율\n'
        f'(Cramér\'s V={cramers_v}, p={p:.4f} / 회색=데이터 부족(30개 미만))',
        fontsize=12
    )
    ax.set_xlabel('마케팅 목적', fontsize=10)
    ax.set_ylabel('콘텐츠 유형', fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

    plt.tight_layout()
    plt.show()

    return ratio_pivot_masked

In [ ]:
# 함수 실행
final_fnb_success_marketing = analyze_content_type_x_marketing_purpose(df_success, '성공')

In [ ]:
# 함수 실행
final_fnb_fail_marketing  = analyze_content_type_x_marketing_purpose(df_fail, '실패')